In [ ]:
import tkinter as tk
import cv2
from PIL import Image, ImageTk
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import load_model

In [ ]:
!ls /dev/video*

In [ ]:
# Validar camara
video = cv2.VideoCapture(0)

print("Abierta:", video.isOpened())

ret, frame = video.read()
print("Leyó frame:", ret)
print("Frame:", None if frame is None else frame.shape)

video.release()

In [ ]:
ALTO = 600
ANCHO = 800

#url = "http://192.168.1.42:4747/video"
#video = cv2.VideoCapture(url)
#video = cv2.VideoCapture(0, cv2.CAP_DSHOW)
video = cv2.VideoCapture(0, cv2.CAP_V4L2)
video.set(cv2.CAP_PROP_FRAME_WIDTH, ANCHO)
video.set(cv2.CAP_PROP_FRAME_HEIGHT, ALTO)

# Detector de rostros
detectorFacial = cv2.CascadeClassifier('haarcascade_frontalface_alt.xml')
#detectorPerfil = cv2.CascadeClassifier("haarcascade_profileface.xml")

clasificadorEmocion = load_model('models/modeloPropio_v1.keras')

gui = tk.Tk()
gui.title("Interfaz de Visión Artificial")
gui.geometry('670x580')

lbl = tk.Label(gui)
lbl.grid(row = 0, column = 0)

In [ ]:
clase2nombre = {0: 'enojo', 1: 'sorpresa', 2: 'disgusto', 3: 'miedo', 4: 'neutro', 5: 'feliz', 6: 'triste', 7: 'desprecio'}

In [ ]:
def abrirCamara():
    ret, frame = video.read()

    if not ret or frame is None:
        print("CAMARA DESCONECTADA")
        lbl.after(50, abrirCamara)
        return

    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # AQUÍ VA EL CÓDIGO DE VISIÓN
    rostros = detectorFacial.detectMultiScale(img, scaleFactor = 1.2, minNeighbors = 5)
    #rostros_perfil = detectorPerfil.detectMultiScale(img, scaleFactor = 1.2, minNeighbors = 5)

    if len(rostros) > 0:
        for x,y,w,h in rostros:
            cv2.rectangle(img, (x,y), (x+w, y+h), color = (0,255,0), thickness = 15)
            rostro = img[y:y+h, x:x+w]
            rostro = cv2.resize(rostro, (117,119))
            rostro = rostro/255
            pred = clasificadorEmocion.predict(rostro.reshape(1,119,117,3))
            clase = np.argmax(pred)
            emocion = clase2nombre[clase]
            cv2.putText(img, emocion, (x,y-10), cv2.FONT_HERSHEY_COMPLEX, fontScale=3, color=(0,255,0),thickness=2)

    img_array = Image.fromarray(img)
    img_photo = ImageTk.PhotoImage(img_array)
    lbl.photo_image = img_photo
    lbl.configure(image = img_photo)
    lbl.after(50, abrirCamara)

def cerrarCamara():
    gui.quit()
    video.release()
    gui.destroy()
    print("CAMARA DESCONECTADA")

In [ ]:
btnAbrir = tk.Button(gui, text = 'Abrir cámara',
                     command = abrirCamara)
btnAbrir.grid(row = 1, column = 0)

btnCerrar = tk.Button(gui, text = 'Cerrar cámara',
                      command = cerrarCamara)
btnCerrar.grid(row = 2, column = 0)

gui.mainloop()